# 工作坊 Solutions Notebook

> ⚠️ **僅供講師對答案 / 課後自學參考——學員應該先試主 notebook**

本 notebook 跟主 notebook 結構完全一樣,差別在於三個「動手時間」的 TODO 已經填上參考解答:
1. §4.6 把 echo 改成 upper-case
2. §8.8 打開 messages list 看裡面結構
3. §12.7 加上第三個 eval case (處理缺失值)

課後練習 Level 1–3 也建議自己先寫過再來對照。

# LangGraph Harness Workshop
## 兩小時打造你的第一個 Data Analyst Agent

> **台大 AI 社・2 小時實作工作坊**

兩小時後你會自己寫出一個 agent——丟它一份 CSV,它會自己決定要先檢查資料、再 groupby、最後寫出結論。更重要的是,你會親手體驗業界從 prompt → context → workflow → **harness** 的演進,並學會用 trace 跟 eval 把 agent 的行為從「神祕」變成「可工程化迭代」。

**受眾預設**:你會基礎 Python (function、list/dict、if/for),但**沒寫過裝飾器、沒呼叫過 LLM API、沒用過 LangGraph**。

**怎麼用這份 notebook**:
- 每個 section 開頭的 markdown 解釋觀念
- 圖示意義:📚 觀念說明・🧪 講師 demo・🛠 動手時間・🔧 預寫 helper (不用看實作)・⚠️ 容易踩的雷・💡 提示
- Helper functions 都已經預先寫在「🔧」cell 裡——你**只需要 run、不用敲**


## §1 環境救火

下面三個 cell 依序 run:
1. 設定 API key (Colab 從 Secrets 讀,本地會跳出輸入框)
2. 安裝相依套件 (Colab 用)
3. 下載故意有缺陷的 `sales.csv`
4. 驗證 LLM 連得上


In [ ]:
# 第一步:設定 API key
# - 在 Colab:從左側「鑰匙」圖示新增 secret 名稱 ANTHROPIC_API_KEY
# - 在 local Jupyter:會跳出輸入框讓你貼 key
import os
import sys


def setup_api_key() -> None:
    """設定 ANTHROPIC_API_KEY,Colab 跟 local 都能用。"""
    # 已經設定過就跳過
    if "ANTHROPIC_API_KEY" in os.environ and os.environ["ANTHROPIC_API_KEY"]:
        print("ANTHROPIC_API_KEY 已從環境變數讀取")
        return

    # 偵測是不是在 Colab
    in_colab = "google.colab" in sys.modules
    if in_colab:
        try:
            from google.colab import userdata  # ty: ignore[possibly-missing-import]

            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
            print("ANTHROPIC_API_KEY 已從 Colab Secrets 讀取")
            return
        except Exception as e:
            print(f"從 Colab Secrets 讀取失敗:{e}")
            print("    改用 getpass 手動輸入...")

    # Local Jupyter 或 Colab Secrets 沒設好,fallback 到手動輸入
    import getpass

    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("請貼上你的 Anthropic API Key: ")
    print("ANTHROPIC_API_KEY 已設定")


setup_api_key()


In [ ]:
# 在 Colab 環境下安裝依賴。Local 用戶請先執行 `uv sync`,可跳過此 cell。
import sys

if "google.colab" in sys.modules:
    %pip install -q -U langgraph langchain langchain-anthropic pandas typing-extensions


In [ ]:
# 下載工作坊使用的 sales.csv
# 這份資料故意有缺失值、混亂日期、重複 row——後半場會用到這些缺陷
import os
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/AluminumShark/langgraph-harness-workshop/main/data/sales.csv"
DATA_PATH = "sales.csv"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"已下載 {DATA_PATH}")
else:
    print(f"{DATA_PATH} 已存在")


In [ ]:
# 確認 LLM 連得上
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-5-20250929")
print(llm.invoke("say hi in 5 words").content)


## §2 LangGraph 心智模型

> 📚 **一句話**:**LangGraph 把 AI 流程畫成一張圖**。

- **節點 (Node)**:一個步驟 (例如「檢查資料」「呼叫 LLM」)
- **邊 (Edge)**:步驟之間的流向
- **狀態 (State)**:節點之間傳遞的資料包

```
START → load_csv → analyze → END
```

這就是 LangGraph 的全部。

## §3 Python 速補:兩個你必須先看過的東西

### §3.1 TypedDict 是什麼

> 📚 **一句話**:TypedDict 是一個**長得像 class 的 dict 設計圖**——告訴別人「這個 dict 應該有哪些 key、各是什麼型別」。

重點三件事:
1. **它就是 dict**——不是真的 class,沒有 `self`、沒有 method,存取用 `alice["name"]` 不是 `alice.name`
2. **typing_extensions 是 typing 的「進階版」**——某些新功能 typing 還沒有,要從 typing_extensions 拿
3. **零 runtime 開銷**——只是給編輯器跟 LangGraph 看的提示,跑起來跟普通 dict 完全一樣


In [ ]:
from typing_extensions import TypedDict


# 設計圖:一個 User dict 應該有 name 和 age
class User(TypedDict):
    name: str
    age: int


# 實際使用就是普通的 dict
alice: User = {"name": "Alice", "age": 20}
print(alice["name"])  # "Alice"


### §3.2 `Annotated` (先預告,§7 才會用到)

> 📚 **一句話**:`Annotated` 讓你給型別**夾帶額外資訊**。Python 本身不會用這資訊,但**框架 (例如 LangGraph) 可以讀**。

等下 §7 你會看到 `Annotated[list, add_messages]`——意思是『這是 list,但 LangGraph 請用 add_messages 這個函數來合併它』。**現在不懂沒關係,看到再講。**

In [ ]:
from typing_extensions import Annotated

# 把 int 加上一個註解 (這個註解不影響執行,只是給工具讀的)
x: Annotated[int, "這是一個年齡"] = 20
print(x)


## §4 第一個 Graph:echo bot

最小可跑的 graph。**只教 State / Node / compile / invoke 四件事,不教 tool、不教 routing**。

Node function 的固定簽名:`state -> dict`
- 吃 state (一個 dict)
- return 一個 dict——LangGraph 會把它**自動 merge 進 state**
- 所以你只 return 你要更新的欄位就好,**不用每次抄整個 state**

> ⚠️ **第一個會踩的雷**:直接寫 `builder.invoke()` 失敗——一定要先 `builder.compile()`。

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. 定義 State:節點之間傳什麼資料
class State(TypedDict):
    user_input: str
    response: str


# 2. 定義 Node:function 簽名一定是 state -> dict
def echo(state: State) -> dict:
    return {"response": f"你說的是:{state['user_input']}"}


# 3. 組裝 graph
builder = StateGraph(State)
_ = builder.add_node("echo", echo)
_ = builder.add_edge(START, "echo")
_ = builder.add_edge("echo", END)

# 4. compile 後才能 invoke
graph = builder.compile()

# 5. 跑
result = graph.invoke({"user_input": "你好"})
print(result)


### §4.6 動手時間 (5 分鐘)

> 🛠 **動手任務**:把 echo 改成「**把使用者輸入轉成大寫**」再回傳。
>
> 提示:`str.upper()` 把字串變大寫。
>
> 寫完跑出來舉手。

In [ ]:
# 解答:upper echo bot
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class UpperState(TypedDict):
    user_input: str
    response: str


def upper_echo(state: UpperState) -> dict:
    return {"response": state["user_input"].upper()}


builder = StateGraph(UpperState)
_ = builder.add_node("upper", upper_echo)
_ = builder.add_edge(START, "upper")
_ = builder.add_edge("upper", END)
upper_graph = builder.compile()
print(upper_graph.invoke({"user_input": "hello"}))
# {'user_input': 'hello', 'response': 'HELLO'}


## §5 兩個節點 + 邊

> 🧪 **這節是 demo,不用動手**——結構跟 §4 幾乎一樣,節省認知能量留給後面 tool calling。

**重點 (用「水管」比喻)**:
- 兩個節點之間**完全不直接互相呼叫**——只透過 state 溝通
- `clean` 把處理結果寫進 state 的 `cleaned` 欄位,`respond` 從 state 讀 `cleaned`
- **state 是節點之間唯一的水管**

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class TwoNodeState(TypedDict):
    user_input: str
    cleaned: str
    response: str


def clean(state: TwoNodeState) -> dict:
    return {"cleaned": state["user_input"].strip().lower()}


def respond(state: TwoNodeState) -> dict:
    return {"response": f"處理後:{state['cleaned']}"}


builder = StateGraph(TwoNodeState)
_ = builder.add_node("clean", clean)
_ = builder.add_node("respond", respond)
_ = builder.add_edge(START, "clean")
_ = builder.add_edge("clean", "respond")
_ = builder.add_edge("respond", END)

two_node_graph = builder.compile()
result = two_node_graph.invoke({"user_input": "  HELLO  "})
print(result)


## §6 LLM 物件解剖:先打開黑箱再進 agent

> 🧪 **這節整段都是 print 給你看**——LLM 內部我們不討論,但**送進去什麼、拿出來什麼**這些都是 Python 物件,可以 print、可以 inspect。

**重點解剖**:
1. `llm.invoke(messages)` 吃一個 **list of dicts**,每個 dict 有 `role` (誰說的) 和 `content` (說了什麼)
2. **回傳是一個 `AIMessage` 物件**——不是字串!要拿文字內容用 `.content`
3. Message 物件還有 `.tool_calls`、`.id` 等屬性,等下會看到


In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-5-20250929")

response = llm.invoke([
    {"role": "user", "content": "你好"}
])

# 注意 type 是 AIMessage,不是 str
print(type(response))

# 拿文字內容用 .content
print(response.content)


### §6.2 多輪對話:messages 是 list

> 💡 **重要事實**:**LLM 沒有記憶**——你每次 invoke 要把整段歷史傳進去,它才知道剛才聊過什麼。

這個事實對等下 §7 理解 `MessagesState` 至關重要:為什麼要 append 不要覆蓋?因為 LLM 需要看完整歷史。

In [ ]:
response = llm.invoke([
    {"role": "user", "content": "我叫小明"},
    {"role": "assistant", "content": "你好,小明!"},
    {"role": "user", "content": "我叫什麼名字?"},
])
print(response.content)


## §7 MessagesState:對話 agent 的標準寫法

**LangGraph 內建 State,內容大概是**:

```python
class MessagesState(TypedDict):
    messages: Annotated[list, add_messages]
```

現在 §3.2 預告的 `Annotated` 派上用場了——這個寫法的意思是:

> 「這欄位是個 list,但 LangGraph 處理它的時候**不要覆蓋**,要用 `add_messages` 這個函數來合併 (也就是 append)。」

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-5-20250929")


def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
_ = builder.add_node("chatbot", chatbot)
_ = builder.add_edge(START, "chatbot")
_ = builder.add_edge("chatbot", END)

graph = builder.compile()
result = graph.invoke({
    "messages": [{"role": "user", "content": "幫我寫一句 Python hello world"}]
})
print(result["messages"][-1].content)


### §7.2 為什麼 messages 要 append 不能覆蓋——現場 demo

> 🧪 **講師 demo**:直接給你看不用 `add_messages` 會發生什麼。

> 💡 **看到 bug 比聽抽象解釋有效十倍。**


In [ ]:
# 反例:自己寫的 State,沒用 add_messages
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class BadState(TypedDict):
    messages: list  # ← 注意這裡會 bug:預設行為是「覆蓋」


def bad_chatbot(state: BadState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


bad_builder = StateGraph(BadState)
_ = bad_builder.add_node("chatbot", bad_chatbot)
_ = bad_builder.add_edge(START, "chatbot")
_ = bad_builder.add_edge("chatbot", END)
bad_graph = bad_builder.compile()

# 跑兩輪:第一輪丟自我介紹,第二輪接著問名字
r1 = bad_graph.invoke({"messages": [{"role": "user", "content": "我叫小明"}]})
print("第一輪 messages 數量:", len(r1["messages"]))  # 怪怪的——輸入應該還在啊

r2 = bad_graph.invoke({
    "messages": r1["messages"] + [{"role": "user", "content": "我叫什麼名字?"}]
})
print(r2["messages"][-1].content)  # 預期:LLM 回「我不知道你的名字」——第一輪訊息不見了


**Bug 的原因**:`return {"messages": [response]}` 把整個 messages list **覆蓋**成只有 response 那一個。原本的 user message 不見了。

**正確版**就是上面的 `MessagesState`——`add_messages` 會幫我們 append 而不是覆蓋。

### §7.3 永遠 return 新 list,不要直接 mutate state

```python
# 錯
def bad_node(state):
    state["messages"].append(new_msg)
    return state

# 對
def good_node(state):
    return {"messages": [new_msg]}
```

> 💡 **永遠不要直接 mutate state**——讓 LangGraph 的 reducer (也就是 `add_messages`) 幫你處理 merge。

## §8 Tool Calling:讓 agent 能呼叫工具

到這裡你會寫對話 bot 了。但 bot 不是 agent——**agent 會自己決定要呼叫工具**。

### §8.1 LLM 怎麼決定呼叫 tool

1. 你把 tool 的「使用說明書」(function 名稱 + 參數 + docstring) 打包成 schema,**透過 API 一起傳給 LLM**
2. LLM 看到使用者的問題,**決定要不要用 tool**:
   - 不用 → 直接回文字
   - 要用 → 回傳一個**特殊格式的訊息**,內容是「請呼叫 `inspect_data('sales.csv')`」
3. 你的程式 (不是 LLM!) **讀到這個特殊訊息,去執行對應的 function**
4. 把執行結果**再餵回給 LLM**,讓它繼續思考

> 💡 **LLM 自己不會執行 Python 程式碼**——它只是「建議」要呼叫哪個 function。實際執行的人是你 (或 LangGraph 的 ToolNode)。**這個觀念極端重要**。

**ReAct loop**:
```
使用者問問題 → LLM 思考 → 要不要呼叫 tool?
                            ├─ 不要 → 直接回答 → END
                            └─ 要 → 呼叫 tool → 看到結果 → 再回去 LLM 思考
```

### §8.2 實際看 LLM 怎麼回 tool_call (進 LangGraph 之前先 demo)

> 🧪 **講師 demo**——讓你親眼看到「LLM 回的 tool_call 物件長什麼樣」,後面看 ToolNode 自動處理時才不黑箱。

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """查詢城市的天氣。輸入城市名,回傳天氣描述。"""
    return f"{city} 今天晴天,25 度"


# 把 tool 綁到 LLM 上 (附上「使用說明書」)
llm_with_tools = llm.bind_tools([get_weather])

response = llm_with_tools.invoke([
    {"role": "user", "content": "台北天氣怎樣?"}
])

# 注意是空字串!LLM 不打算用文字回答,而是建議我們呼叫 tool
print(repr(response.content))

# tool_calls 是 list,每個元素是 dict {name, args, id}
print(response.tool_calls)


**重點解剖**:
- LLM 決定用 tool 時,`response.content` 通常是**空的**
- `response.tool_calls` 是一個 list,每個元素是 dict:`{"name": tool 名字, "args": 參數, "id": 識別碼}`
- **LLM 沒有真的執行 `get_weather`!** 它只是建議我們去呼叫
- 我們要拿這個 dict 去**自己呼叫**對應的 Python function

下面 §8.4 LangGraph 的 `ToolNode` 會幫我們自動處理「讀 tool_calls → 呼叫 function → 把結果包回去」這一連串 boilerplate。

### §8.3 `@tool` 裝飾器:30 秒定義盒

**裝飾器是什麼**:`@tool` 是把 function「包裝」一層的語法。原本 `get_weather` 只是普通 function,加了 `@tool` 之後它變成一個 LangChain 的 Tool 物件,多了「告訴 LLM 我長怎樣」的能力。

**docstring 為什麼重要**:LLM **完全靠 docstring 決定要不要呼叫這個 tool**。docstring 寫不清楚 = LLM 永遠不呼叫。好的 docstring 要交代:做什麼、什麼時候該呼叫、參數意義、回傳什麼。

### §8.4 完整實作 Data Analyst Agent

下面整段是這節最重要的程式——把它跑起來,你就有了一個會自己分析 CSV 的 agent。

> ⚠️ **本工作坊刻意設計**:這個 agent 沒有 system prompt。**等下你跑起來行為可能不太穩**——這是後半場 §12 要修的伏筆,**現在不要急著加 prompt**。

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, MessagesState
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
import pandas as pd


@tool
def inspect_data(path: str) -> str:
    """檢查 CSV 資料品質:欄位、缺失值、樣本。

    分析 CSV 之前一定要先呼叫這個工具,確認資料的形狀、欄位名稱、
    哪些欄位有缺失值、頭幾筆長什麼樣。
    """
    df = pd.read_csv(path)
    return (
        f"Shape: {df.shape}\n"
        f"Columns: {list(df.columns)}\n"
        f"Missing: {df.isna().sum().to_dict()}\n"
        f"Head:\n{df.head().to_string()}"
    )


@tool
def run_pandas(path: str, code: str) -> str:
    """執行 pandas 程式碼。code 內 df 變數已經是讀好的 DataFrame。

    用來實際做 groupby、filter、agg 等運算。
    """
    df = pd.read_csv(path)
    safe_globals = {"df": df, "pd": pd}
    try:
        result = eval(code, safe_globals)
        return str(result)[:2000]
    except Exception as e:
        return f"Error: {e}"


tools = [inspect_data, run_pandas]
llm = ChatAnthropic(model="claude-sonnet-4-5-20250929").bind_tools(tools)


def chatbot(state: MessagesState) -> dict:
    # 注意:這裡刻意沒加 system prompt——後半場 §12 才會加上
    return {"messages": [llm.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
_ = builder.add_node("chatbot", chatbot)
_ = builder.add_node("tools", ToolNode(tools))
_ = builder.add_edge(START, "chatbot")
_ = builder.add_conditional_edges("chatbot", tools_condition)
_ = builder.add_edge("tools", "chatbot")
graph = builder.compile()


### §8.6 圖示意

```
START → chatbot ──tool_calls 是空─→ END
            │
            └─有 tool_calls─→ tools ─→ (回到 chatbot)
```

### §8.7 試跑

> 🧪 **跑跑看**——結果可能 East、可能 West、可能不一致。**這是正常的**,不要急著修。

In [ ]:
result = graph.invoke({
    "messages": [{"role": "user", "content": "幫我分析 sales.csv,找出表現最差的區域"}]
})
print(result["messages"][-1].content)


### §8.8 動手時間 (10 分鐘)

> 🛠 **動手任務**:
> 1. 上面的 graph 完整跑過一次
> 2. **重點任務**:把 `result["messages"]` print 出來看裡面有幾個 message、各是什麼 type、`.tool_calls` 有沒有東西

這個任務是刻意的——**強迫你把 messages list 打開來看**,內化「LangGraph 內部到底是怎麼跑的」。

In [ ]:
# 解答:打開 messages 看裡面
result = graph.invoke({
    "messages": [{"role": "user", "content": "幫我分析 sales.csv,找出表現最差的區域"}]
})

for i, msg in enumerate(result["messages"]):
    print(f"--- message[{i}] ---")
    print(f"  type: {type(msg).__name__}")
    print(f"  content: {repr(msg.content)[:120]}")
    tcs = getattr(msg, "tool_calls", None)
    if tcs:
        print(f"  tool_calls: {[(tc['name'], tc['args']) for tc in tcs]}")


## §9 前半場小結 + 主動破除自我懷疑

> 💡 **重要的話**——剛才你跑出來的 agent,結果可能不太一樣:
> - 有人跑出 **East** 是表現最差的
> - 有人跑出 **West**
> - 有人跑兩次答案不一樣
>
> **這不是你寫錯了**。這是 LLM agent 的**正常變異**——同一個程式、同一份資料,每次跑可能走不同路徑、做不同決定。
>
> 這是 LLM 的本質特性,**業界所有人都在處理這件事**。後半場我們就要學業界怎麼處理:
> 怎麼**看見** agent 的決策 (trace)、怎麼**量化** agent 的好壞 (eval)。

---

### 中場休息 5 分鐘

# 後半場:把 agent 變成可演進的工程

## §10 後半場開場:四階段演進敘事

### §10.1 大家剛才都遇到了什麼

> 🧪 **舉手調查 (講師主持)**:
> - 剛剛 agent 跑出「最差是 East」的舉手 → 一些人舉
> - 跑出「最差是 West」的舉手 → 一些人舉
> - 兩次跑出來不一樣的舉手 → 又一些人舉

**同一個程式、同一份資料、不同人跑出不同答案。這是 LLM agent 的本質。問題是:你怎麼處理這件事?**


### §10.2 業界四階段演進

```
2022──────2023──────2024──────2025──────2026──→
prompt    context   workflow  harness
engineer  engineer  engineer  engineer
```

| 階段 | 核心問題 | 代表性實踐 | 為什麼不夠 |
|------|---------|-----------|-----------|
| **Prompt engineering** (2022–2023) | 「怎麼問 LLM 才聽話」 | few-shot、CoT、role prompting | 單輪、無狀態、無工具 |
| **Context engineering** (2023–2024) | 「怎麼把對的資訊塞進 context」 | RAG、長 context、memory | 仍是「一次問答」,無流程 |
| **Workflow engineering** (2024–2025) | 「怎麼把多步流程結構化」 | LangGraph、DAG agent、multi-agent | 跑得起來但**無法演進、無法量化** |
| **Harness engineering** (2025–) | 「怎麼讓 agent 可被工程化迭代」 | eval set、trace、replay、observability | ← **本工作坊的位置** |

**你前半場寫的是 workflow engineering 的標準作品**——LangGraph 是這個階段的代表工具。但業界已經知道這樣不夠了。


### §10.3 Harness 是什麼

**Harness** (馬具) 這個詞來自工程實踐:你給 LLM 套上一組「可控制、可觀測、可評估」的裝備,讓 agent 行為從**藝術品**變成**工業品**。

完整 harness 包含四件事:

| 元件 | 它讓你回答的問題 | 今天會教嗎 |
|------|---------------|-----------|
| **Trace** | 「這次 agent 為什麼這樣決策?」 | 是 §11 |
| **Replay** | 「失敗時能不能從中間試?」 | 課後練習 (今天時間不夠) |
| **Eval** | 「我改了 prompt,整體變好還是變壞?」 | 是 §12 |
| **Loop** | 「怎麼把上面接成持續改進的迴圈?」 | 是 §13 |

## §11 Trace:讓 agent 的每一步可觀測

### §11.1 問題:剛才到底發生了什麼?

用 `graph.invoke()` 你只看得到最後一句答案。中間 agent 呼叫了哪些 tool、看到什麼結果——**全部不可見**。

### §11.2 解法:用 `stream` 取代 `invoke`

LangGraph 提供 `stream` 介面——跟 `invoke` 一樣的輸入,但會**把每個節點 return 的東西即時吐出來**。

**`stream_mode="updates"` 的意思**:每個節點跑完後,把『它 return 了什麼』吐出來。`events` 是一個 list,每個元素是 `{節點名字: 那個節點 return 的 dict}`。

In [ ]:
events = list(graph.stream(
    {"messages": [{"role": "user", "content": "分析 sales.csv,找出表現最差的區域"}]},
    stream_mode="updates",
))

# 先 print 出來看一下原始結構
print(events[0])
# 大概像:{'chatbot': {'messages': [AIMessage(content='', tool_calls=[...])]}}


### §11.3 把 trace 包成漂亮的 print

> 🔧 **Helper cell:`pretty_trace` (你不用敲,直接 run 即可)**
>
> 這個 helper 等同於 `src/langgraph_harness_workshop/helpers.py` 的內容,內聯放在 notebook 方便 Colab 學員。
> **重點是會用,不是會寫。**

In [ ]:
def pretty_trace(graph, user_message: str) -> None:
    """跑 graph,把每個節點的決策漂亮印出來。"""
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    for event in graph.stream(inputs, stream_mode="updates"):
        for node_name, update in event.items():
            print(f"\n[{node_name}]")
            for msg in update.get("messages", []):
                # 如果這個 message 帶 tool_calls,印出工具呼叫
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"  → 呼叫工具: {tc['name']}({tc['args']})")
                # 如果這個 message 有文字內容,截斷到 150 字印出來
                if hasattr(msg, "content") and msg.content:
                    print(f"  → 訊息: {msg.content[:150]}")


現在你只需要呼叫 `pretty_trace(graph, ...)`,就能看到 agent 每一步的決策。

In [ ]:
pretty_trace(graph, "分析 sales.csv,找出表現最差的區域")


### §11.4 回到舉手調查的問題

**「為什麼兩個人跑同樣的問題結果不一樣?」**

對比兩個學員的 trace:
- A 同學:`[chatbot → inspect_data → chatbot → run_pandas → chatbot]` ✓
- B 同學:`[chatbot → run_pandas → chatbot]` (跳過 inspect) ✗

現在你知道差在哪了——B 同學的 LLM 決定不呼叫 `inspect_data`,所以沒看到 NaN,groupby 結果錯了。

> 💡 **這就是 trace 的價值**:問題從「神祕」變成「具體可指認的步驟差異」。

### §11.5 動手時間 (5 分鐘)

> 🛠 **動手任務**:用 `pretty_trace` 跑**三次**同樣的問題,看 agent 三次行為一不一樣。

In [ ]:
# 動手時間:跑三次,觀察行為差異
for i in range(3):
    print(f"\n========== 第 {i+1} 次 ==========")
    pretty_trace(graph, "分析 sales.csv,找出表現最差的區域")


## §12 Eval:用數字說話 (後半場最重要)

### §12.1 問題:改了 prompt 之後不知道變好變壞

假設我覺得剛才 agent 表現不夠好,我想加一句 system prompt:「記得先 inspect 再 groupby」。
**改完之後我怎麼知道是真的變好了?**

> 💡 **慘痛事實**:很多人改 prompt 是靠「感覺好像比較好」。這就是為什麼 LLM 產品難以演進——你沒有客觀基準。

### §12.2 解法:寫 eval case

**Eval case 就是『我認為 agent 應該怎麼做』的機械版本**——把標準寫成程式可以檢查的規則。

In [ ]:
EVAL_CASES = [
    {
        "id": "001_must_inspect_first",
        "query": "找出表現最差的區域",
        "must_call_tools_in_order_prefix": ["inspect_data", "run_pandas"],
        # 第一個工具呼叫一定要是 inspect_data
        "answer_must_not_contain": ["NaN", "nan"],
        # 最終答案不能包含 NaN (代表沒處理乾淨)
        "answer_must_contain_one_of": ["East", "West", "North", "South"],
        # 答案要明確指出某個區域
    },
    {
        "id": "002_handle_empty_result",
        "query": "找出 2099 年的資料",
        "answer_must_contain_one_of": ["沒有", "找不到", "no data", "無資料"],
    },
]

print(f"目前有 {len(EVAL_CASES)} 個 eval case")


### §12.3 寫 eval runner

> 🔧 **Helper cell:`extract_tool_calls` / `extract_final_text` / `evaluate_one` / `run_eval_suite` (你不用敲)**
>
> 邏輯就是「跑 graph、抽出 tool 呼叫順序、抽出最終答案、檢查是否符合 case 描述的規則」。
> **重點是會用。**

In [ ]:
def extract_tool_calls(events):
    """從一連串的 stream event 中,依序抽出工具呼叫名稱。"""
    sequence = []
    for event in events:
        for _node, update in event.items():
            for msg in update.get("messages", []):
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        sequence.append(tc["name"])
    return sequence


def extract_final_text(events):
    """從 stream event 中抽出 agent 最終的文字回答。"""
    last_text = ""
    for event in events:
        for _node, update in event.items():
            for msg in update.get("messages", []):
                if hasattr(msg, "content") and msg.content and isinstance(msg.content, str):
                    last_text = msg.content
    return last_text


def evaluate_one(graph, case):
    """跑一個 eval case,回傳 {case_id, passed, checks}。"""
    events = list(graph.stream(
        {"messages": [{"role": "user", "content": case["query"]}]},
        stream_mode="updates",
    ))
    tool_seq = extract_tool_calls(events)
    answer = extract_final_text(events)

    checks = {}
    # 檢查 1:工具呼叫順序前綴比對
    if "must_call_tools_in_order_prefix" in case:
        prefix = case["must_call_tools_in_order_prefix"]
        checks["tool_order"] = tool_seq[:len(prefix)] == prefix
    # 檢查 2:答案不該包含的關鍵字
    if "answer_must_not_contain" in case:
        checks["no_forbidden"] = all(
            kw.lower() not in answer.lower()
            for kw in case["answer_must_not_contain"]
        )
    # 檢查 3:答案至少要包含其中一個關鍵字
    if "answer_must_contain_one_of" in case:
        checks["has_required"] = any(
            kw.lower() in answer.lower()
            for kw in case["answer_must_contain_one_of"]
        )
    return {"case_id": case["id"], "passed": all(checks.values()) if checks else False, "checks": checks}


def run_eval_suite(graph, cases):
    """跑完一整批,回傳 pass_rate 跟細節。"""
    results = [evaluate_one(graph, c) for c in cases]
    pass_rate = sum(r['passed'] for r in results) / len(results) if results else 0.0
    return {"pass_rate": pass_rate, "results": results}


### §12.4 跑第一次 baseline

現場跑——通常會看到 **50–60%** 左右 (因為 agent 沒 system prompt,行為不穩定)。

In [ ]:
report = run_eval_suite(graph, EVAL_CASES)
print(f"Pass rate: {report['pass_rate']:.0%}")
for r in report["results"]:
    status = "✓" if r["passed"] else "✗"
    print(f"  {status} {r['case_id']}: {r['checks']}")


### §12.5 加一行 prompt 看 eval 動 (這節的高潮)

現在我們**改一個地方**——加 system prompt,然後重跑同一份 eval set。

> 💡 **Pass rate 從 ~60% 變 ~90%**——改一行 prompt,**量化看到變好**。這就是工程化迭代的感覺。

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, MessagesState

SYSTEM_PROMPT = """你是資料分析助手。
在執行任何 groupby/filter/agg 操作之前,**必須**先呼叫 inspect_data 檢查資料品質。
如果有缺失值,先在答案中說明你怎麼處理。"""


def chatbot_v2(state: MessagesState) -> dict:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + list(state["messages"])
    return {"messages": [llm.invoke(messages)]}


# 重新組 graph (chatbot 換成 v2)
builder = StateGraph(MessagesState)
_ = builder.add_node("chatbot", chatbot_v2)
_ = builder.add_node("tools", ToolNode(tools))
_ = builder.add_edge(START, "chatbot")
_ = builder.add_conditional_edges("chatbot", tools_condition)
_ = builder.add_edge("tools", "chatbot")
graph_v2 = builder.compile()

# 重跑 eval
report_v2 = run_eval_suite(graph_v2, EVAL_CASES)
print(f"v2 Pass rate: {report_v2['pass_rate']:.0%}")
for r in report_v2["results"]:
    status = "✓" if r["passed"] else "✗"
    print(f"  {status} {r['case_id']}: {r['checks']}")


### §12.6 Eval set 多大才算夠

> 💡 從 1 個開始就比 0 個好。每抓到一個失敗 case 就加一個。**有方向比有數量重要**。
> Anthropic 內部的 eval set 也是從幾十個 case 開始長到幾千個。今天你寫了 2 個——已經比昨天的自己強了。

### §12.7 動手時間 (10 分鐘)

> 🛠 **動手任務**:
> 1. 跑 baseline eval (沒 system prompt 那版,`graph`)
> 2. 跑 v2 eval (有 system prompt 那版,`graph_v2`)
> 3. 看 pass rate 差多少
> 4. **自己想一個第三個 eval case 加進去** (可以模仿前兩個的格式)
>
> 提示:可以挑「處理重複 row」、「型別轉換」、「缺失值處理」等場景。

In [ ]:
# 解答:加上一個處理 NaN 的第三個 eval case
MY_EVAL_CASES = EVAL_CASES + [
    {
        "id": "003_acknowledge_missing_values",
        "query": "sales.csv 裡有多少筆 region 是缺失的?",
        # 答案應該明確提到「8」這個數字
        "answer_must_contain_one_of": ["8", "八"],
        # 不能含 NaN 字面值 (代表沒處理乾淨)
        "answer_must_not_contain": ["NaN"],
    },
]

my_report = run_eval_suite(graph_v2, MY_EVAL_CASES)
print(f"Pass rate: {my_report['pass_rate']:.0%}")
for r in my_report["results"]:
    status = "✓" if r["passed"] else "✗"
    print(f"  {status} {r['case_id']}: {r['checks']}")


## §13 Harness Loop:把全部串起來

```
        ┌─────────────────────────────┐
        │   LangGraph Agent           │
        │   (state + nodes + edges)   │
        └──────────────┬──────────────┘
                       │ invoke
                       ▼
              ┌─────────────────┐
              │ Trace + Checkpoint │  ← stream_mode="updates"
              │ (每步留下證據)        │
              └────────┬────────┘
                       │
            ┌──────────┴──────────┐
            ▼                     ▼
      ┌──────────┐           ┌──────────┐
      │  Replay  │           │   Eval   │
      │ (課後玩)  │           │ (跑 cases) │
      └────┬─────┘           └────┬─────┘
           │                      │
           └──────────┬───────────┘
                      ▼
              ┌──────────────┐
              │  迭代 graph  │   ← 改 prompt / 改 tool / 改路由
              └──────┬───────┘
                     │
                     └──→ 回到頂部
```

今天兩小時你寫的東西已經涵蓋這整個迴圈的核心。**Replay 那塊我們留作課後練習**——LangGraph 的 checkpointer 可以讓你「時光倒流」從任意 checkpoint 重跑,但今天時間不夠,留給你回家玩 (見最後課後練習 Level 3)。

## §14 為什麼是 LangGraph:跟其他框架比起來如何

### §14.1 2026 年的 agent 框架格局 (不是 LangGraph 獨大)

| 框架 | 推手 | 核心抽象 | 定位 |
|------|------|---------|------|
| **LangGraph** | LangChain | 顯式狀態圖 | 跨模型、production mileage 最高 |
| **OpenAI Agents SDK** | OpenAI | Handoff (agent 轉手) | 綁 OpenAI 模型,2026/4 剛加 harness 系統 |
| **Claude Agent SDK** | Anthropic | Tool-use chain + sub-agents | 從 Claude Code SDK 演變,綁 Claude 模型 |
| **CrewAI** | CrewAI Inc. | Role-based crews | 角色化多 agent,上手快 |
| **AWS Strands Agents** | AWS | Model-agnostic + Bedrock 整合 | AWS 生態系 |

### §14.4 該怎麼選

| 你的場景 | 推薦 |
|---------|------|
| 第一次寫 agent / 一個工具 / 不用流程 | **OpenAI 直接 function calling** 三行寫完更好 |
| 已綁 OpenAI 生態系 + 簡單 handoff | **OpenAI Agents SDK** |
| **複雜流程 / 要 trace eval / 跨模型** (今天教的場景) | **LangGraph** |
| Anthropic 模型 + safety-critical | **Claude Agent SDK** |

> 💡 **框架會變,但 harness 思維留下**——一年後你可能在用別的工具,但 trace、eval、迭代閉環會跟著你走。

## §15 Cheatsheet

### §15.1 LangGraph 必踩的 6 個雷

| # | 雷 | 正解 |
|---|-----|------|
| 1 | 直接 `state["messages"].append(...)` | 一律 `return {"messages": [new_msg]}`,讓 reducer 處理 |
| 2 | 自訂 State 沒寫 reducer,訊息列表被覆蓋 | `messages: Annotated[list, add_messages]` |
| 3 | 沒 `compile()` 就 `invoke` | `graph = builder.compile()` 後才能 invoke |
| 4 | `@tool` docstring 沒寫「使用時機」 | LLM 不知何時呼叫 → 永遠不呼叫 |
| 5 | 路由函數裡呼叫 LLM | 路由只讀 state 純值,LLM 放前一個節點 |
| 6 | 用舊 API (`MemorySaver`、`NodeInterrupt`) | 新 API:`InMemorySaver`、`interrupt()` |

### §15.2 Harness 自檢清單

寫 production agent 之前自問:

- [ ] **能 trace 嗎?** 看得到每次每個節點的決策嗎?
- [ ] **State 顯式嗎?** 節點之間傳的資料都在 State schema 裡嗎?
- [ ] **有 eval set 嗎?** 至少 5 個 case 涵蓋已知失敗模式嗎?
- [ ] **改了東西能量化嗎?** 一行 command 跑 eval 出 pass rate 嗎?
- [ ] **改 graph 有紀錄嗎?** 每次改有對應的 eval report 嗎?

> 沒打勾不算寫完 agent,只算寫了個 demo。

## §16 收場

兩小時你學到的不只是 LangGraph,是業界四階段演進的最新一站:

**prompt → context → workflow → harness**

前半場你寫的 agent 是 workflow engineering 的標準作品——這是 2024 的 SOTA。
後半場我們把它升級到 2025–2026 的標準:**trace、eval、迭代閉環**。

> 💡 你今天親身體驗了一件事:**同樣的 agent,加上 harness 之後,從「跑得起來」變成「可以工程化迭代」**。沒有 harness 的 agent 永遠停在 demo 階段。

至於框架——LangGraph 在 2026 年 4 月仍是這個場景最好的選擇,但 OpenAI、Anthropic、AWS 都在追。
**框架會變,但今天教你的不只是 API,是 harness 思維**。一年後你可能在用別的工具,但 trace、eval、loop 這四件事會跟著你走。

## 課後練習

**Level 1 (必做)**:把今天的 eval set 從 2 個 case 擴充到 5 個。涵蓋:缺失值處理、型別轉換、空 DataFrame、重複 row、單一欄位 groupby。

**Level 2**:把今天的 agent 加上 `InMemorySaver` checkpointer,讓多輪對話有記憶。提示:
```python
from langgraph.checkpoint.memory import InMemorySaver
graph = builder.compile(checkpointer=InMemorySaver())
# 之後 invoke 要帶 config={"configurable": {"thread_id": "abc"}}
```

**Level 3**:研究 `graph.get_state_history()` 跟 `graph.update_state()`,實作「從歷史 checkpoint 改一個值再重跑」。這就是 §10.3 預告的 **replay** 機制。

**Level 4**:選一個你做過的個人專案,畫出它的「missing harness」清單——trace 缺什麼、eval 缺什麼。寫成 markdown 帶來下次社課討論。

## 延伸資源

- 官方文件:https://langchain-ai.github.io/langgraph/
- LangGraph Academy (免費):https://academy.langchain.com/courses/intro-to-langgraph
- LangSmith (trace + eval 的 production 平台):https://smith.langchain.com/
- Hamel Husain 〈Your AI Product Needs Evals〉:https://hamel.dev/blog/posts/evals/
- Anthropic 對 agent harness 的工程觀點:https://www.anthropic.com/research/swe-bench-sonnet